# Chapter 2 Federated Analysis

##  Dirichlet Partitioning: How Heterogeneity Distorts Client Distributions

This is the core of the experimental design. For each α value we generate one
partitioning of patients into 5 clients and visualize:
- The positive rate per client (how skewed the label distributions are).
- The Wasserstein distance of each client's key feature distributions from the global.

The Wasserstein distance $W_1$ is the natural measure of distribution divergence here
because it corresponds directly to the $d_{\mathcal{H}\Delta\mathcal{H}}$ term in
Theorem 1 of the FedGen paper — larger $W_1$ predicts worse global model performance.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import wasserstein_distance, entropy, ks_2samp
import warnings
import sys
import os
import json

warnings.filterwarnings('ignore')
sys.path.append('..')

from UC1Utils import (
    load_data, drop_columns, remove_deceased, create_target,
    encode_features, _impute_and_group_specialty,
    prepare_data, ensure_data, CSV_MAIN, derive_global_columns
)
from UC1FLUtils import dirichlet_partition, find_feasible_params, preprocess_clients, verify_leakage, load_clients, create_clients_raw_csv, load_partitions_from_disk

plt.rcParams.update({
    'figure.dpi':       130,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size':        11,
})
PALETTE = {'Not readmitted (0)': '#4878CF', 'Readmitted <30d (1)': '#E24A33'}
ALPHA_SWEEP = [0.1, 0.5, 1.0, 5.0,10.0]
N_CLIENTS   = 5
SEED        = 42
FEDERATED_DIR  = '../federated_data'
FILTERED_DIR   = os.path.join(FEDERATED_DIR, 'filtered')
UNFILTERED_DIR = os.path.join(FEDERATED_DIR, 'unfiltered')

os.makedirs('figures', exist_ok=True)
print('Setup complete.')

In [ ]:
ensure_data()

# ── Load and preprocess up to (but not including) OHE ─────────────────────────
# We keep a pre-OHE DataFrame for clinical interpretation.
df_raw = load_data(CSV_MAIN)
df_raw = drop_columns(df_raw)
df_raw = remove_deceased(df_raw)
df_raw = create_target(df_raw)
df_raw['race'] = df_raw['race'].fillna('Unknown')
df_raw = _impute_and_group_specialty(df_raw)
df_raw['A1Cresult']     = df_raw['A1Cresult'].fillna('none')
df_raw['max_glu_serum'] = df_raw['max_glu_serum'].fillna('none')
df_raw.dropna(inplace=True)

# Apply encode_features which adds engineered columns but also OHEs —
# we'll rebuild a pre-OHE version for visualization.
# The approach: apply everything EXCEPT get_dummies by reconstructing manually.
from UC1Utils import AGE_MAP, ADMISSION_TYPE_MAP, ADMISSION_SOURCE_MAP, MED_MAP, MED_COLS, LOW_INFO_MEDS
from UC1Utils import group_discharge, group_icd9

df = df_raw.copy()
df['age'] = df['age'].map(AGE_MAP)
df['service_utilization'] = df['number_inpatient'] + df['number_outpatient'] + df['number_emergency']
df['admission_type_id']        = df['admission_type_id'].map(ADMISSION_TYPE_MAP).fillna('unknown')
df['admission_source_id']      = df['admission_source_id'].map(ADMISSION_SOURCE_MAP).fillna('unknown')
df['discharge_disposition_id'] = df['discharge_disposition_id'].apply(group_discharge)
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(group_icd9)
for col in MED_COLS:
    if col in df.columns:
        df[col] = df[col].map(MED_MAP)
active_meds = [c for c in MED_COLS if c in df.columns]
df['medication_count'] = df[active_meds].sum(axis=1)
df.drop(columns=[c for c in LOW_INFO_MEDS if c in df.columns], inplace=True)
df['HbA1c_diabetes_interaction'] = (
    (df['A1Cresult'] != 'none') & (df['diag_1'] == 'diabetes')
).astype(int)
df['change']      = (df['change'] == 'Ch').astype(int)
df['diabetesMed'] = (df['diabetesMed'] == 'Yes').astype(int)

# ── Fully encoded matrix ───────────────────────────────────────────────────────
X, y, groups, feature_names = prepare_data(CSV_MAIN, verbose=True)

df_enc = pd.DataFrame(X, columns=feature_names)
df_enc['readmitted_binary'] = y

print(f'\nPre-OHE DataFrame: {df.shape}')
print(f'Encoded matrix:    {df_enc.shape}')

In [ ]:
# Build patient → label map from the full dataset
df_full_raw = pd.read_csv(CSV_MAIN)
patient_labels_map = (
    df_full_raw.groupby('patient_nbr')['readmitted']
    .apply(lambda x: 1 if (x == '<30').any() else 0)
    .to_dict()
)
unique_patients = np.array(list(patient_labels_map.keys()))

print(f'Unique patients: {len(unique_patients):,}')
print(f'Positive patients (ever readmitted <30d): {sum(patient_labels_map.values()):,}')

In [ ]:
# ── Federation Setup: paths and global feature schema ─────────────────────────
# Schema is derived once from the full dataset and saved to disk.
# All client preprocessing aligns to this schema so every client produces
# identical feature dimensions regardless of which rare categories appear locally.

os.makedirs(FEDERATED_DIR, exist_ok=True)

global_columns = derive_global_columns(CSV_MAIN)

schema_path = os.path.join(FEDERATED_DIR, 'feature_schema.json')
with open(schema_path, 'w') as f:
    json.dump(global_columns, f, indent=2)

print(f'Global schema: {len(global_columns)} columns saved → {schema_path}')

In [ ]:
find_feasible_params(df_full_raw, n_clients_list=[5], alpha_list=[0.5, 1.0, 5.0, 10.0])

In [ ]:
new_data = False
PARTITION_SEED = 42

# ── Filtered ──────────────────────────────────────────────────────────────────
if new_data:
    filtered_partitions = create_clients_raw_csv(
        df_full_raw, FILTERED_DIR, ALPHA_SWEEP,
        filtered=True, seed=PARTITION_SEED
    )
    preprocess_clients(
        FILTERED_DIR, ALPHA_SWEEP, global_columns,
        seed=PARTITION_SEED, regenerate=True
    )
else:
    filtered_partitions = load_partitions_from_disk(
        FILTERED_DIR, ALPHA_SWEEP, patient_labels_map
    )
verify_leakage(ALPHA_SWEEP, output_dir=FILTERED_DIR)

# ── Unfiltered ────────────────────────────────────────────────────────────────
if new_data:
    unfiltered_partitions = create_clients_raw_csv(
        df_full_raw, UNFILTERED_DIR, ALPHA_SWEEP,
        filtered=False, seed=PARTITION_SEED
    )
    preprocess_clients(
        UNFILTERED_DIR, ALPHA_SWEEP, global_columns,
        seed=PARTITION_SEED, regenerate=True
    )
else:
    unfiltered_partitions = load_partitions_from_disk(
        UNFILTERED_DIR, ALPHA_SWEEP, patient_labels_map
    )
verify_leakage(ALPHA_SWEEP, output_dir=UNFILTERED_DIR)

## Wasserstein Distance: Client vs. Global Distribution

For each (α, client, feature) triplet, we compute the 1-Wasserstein distance between
the client's feature distribution and the global distribution. This quantifies how
much the Dirichlet split distorts each feature's distribution, and for which features
the distortion is strongest.

This directly operationalizes the $d_{\mathcal{H}\Delta\mathcal{H}}$ term from
FedGen's Theorem 1: clients with high Wasserstein distances on discriminative features
are the ones that will benefit most from the generator's inductive bias.

## Filtered


In [ ]:
WASS_FEATURES = [
    'time_in_hospital', 'number_inpatient', 'number_emergency',
    'service_utilization', 'medication_count', 'num_medications', 'age'
]

global_distributions = {feat: df[feat].values for feat in WASS_FEATURES}

wass_records = []

for alpha, partition in filtered_partitions.items():
    for client_id, patient_ids in partition.items():
        pid_set = set(patient_ids)
        client_df = df[df['patient_nbr'].isin(pid_set)] if 'patient_nbr' in df.columns else df.iloc[:0]
        
        for feat in WASS_FEATURES:
            if feat in df.columns:
                client_vals = df.loc[df['patient_nbr'].isin(pid_set), feat].values if 'patient_nbr' in df.columns else np.array([])
                if len(client_vals) > 10:
                    w = wasserstein_distance(global_distributions[feat], client_vals)
                    wass_records.append({
                        'alpha': alpha, 'client': f'C{client_id}',
                        'feature': feat, 'wasserstein': w
                    })

wass_df_filtered = pd.DataFrame(wass_records)
print(wass_df_filtered.groupby(['alpha', 'feature'])['wasserstein'].mean().unstack('feature').round(4))

In [ ]:
if not wass_df_filtered.empty:
    fig, axes = plt.subplots(1, len(ALPHA_SWEEP), figsize=(5 * len(ALPHA_SWEEP), 5), sharey=True)
    if len(ALPHA_SWEEP) == 1:
        axes = [axes]
    fig.suptitle('Wasserstein Distance: Client vs Global Feature Distribution',
                 fontweight='bold', fontsize=13)

    for ax, alpha in zip(axes, ALPHA_SWEEP):
        sub = wass_df_filtered[wass_df_filtered['alpha'] == alpha]
        if sub.empty:
            ax.set_visible(False); continue
        pivot = sub.pivot(index='feature', columns='client', values='wasserstein')
        pivot.plot(kind='barh', ax=ax, colormap='Set2', alpha=0.85, edgecolor='white')
        ax.set_title(f'α = {alpha}', fontsize=12, fontweight='bold')
        ax.set_xlabel('W₁ distance from global')
        ax.set_ylabel('Feature' if ax == axes[0] else '')
        ax.legend(title='Client', fontsize=8, loc='lower right')

    plt.tight_layout()
    plt.savefig('figures/07_wasserstein_distances.png', bbox_inches='tight')
    plt.show()

    # ── Mean W1 per alpha ──────────────────────────────────────────────────────
    # NOTE: W₁ is NOT monotone in α for this dataset.
    # α is a Dirichlet draw parameter — any single realization may not respect
    # the expected heterogeneity ordering. W₁ measures the actual realized
    # divergence and is the reliable metric for the Pareto analysis.
    mean_wass = wass_df_filtered.groupby('alpha')['wasserstein'].mean()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(mean_wass.index, mean_wass.values, 'o-', color='steelblue',
            linewidth=2, markersize=8)

    # Annotate each point with its realized W₁ value
    for alpha_val, w1_val in mean_wass.items():
        ax.annotate(f'W₁={w1_val:.3f}',
                    xy=(alpha_val, w1_val),
                    xytext=(6, 6), textcoords='offset points',
                    fontsize=8, color='steelblue')

    ax.set_xlabel('Dirichlet α  (generation parameter, not realized heterogeneity)')
    ax.set_ylabel('Mean W₁ (across clients and features)')
    ax.set_title('Realized W₁ heterogeneity per partition\n'
                 'α is not a monotone proxy for W₁ in this dataset',
                 fontweight='bold')
    ax.invert_xaxis()

    # Identify and annotate the actual most heterogeneous partition
    most_het_alpha = mean_wass.idxmax()
    ax.annotate(f'Most heterogeneous\nrealized partition',
                xy=(most_het_alpha, mean_wass[most_het_alpha]),
                xytext=(-60, -30), textcoords='offset points',
                fontsize=8, color='#E24A33',
                arrowprops=dict(arrowstyle='->', color='#E24A33', lw=1.2))

    plt.tight_layout()
    plt.savefig('figures/08_mean_wasserstein_vs_alpha.png', bbox_inches='tight')
    plt.show()

    print('Realized heterogeneity ordering by W₁:')
    print(mean_wass.sort_values(ascending=False).to_string())
    print(f'\nNote: α ordering ({sorted(mean_wass.index, reverse=True)}) does not match '
          f'W₁ ordering ({list(mean_wass.sort_values(ascending=False).index)})')
    print('Use W₁ values to anchor the Pareto analysis x-axis, not α alone.')

## Unfiltered

In [ ]:
WASS_FEATURES = [
    'time_in_hospital', 'number_inpatient', 'number_emergency',
    'service_utilization', 'medication_count', 'num_medications', 'age'
]

global_distributions = {feat: df[feat].values for feat in WASS_FEATURES}

wass_records = []

for alpha, partition in unfiltered_partitions.items():
    for client_id, patient_ids in partition.items():
        pid_set = set(patient_ids)
        client_df = df[df['patient_nbr'].isin(pid_set)] if 'patient_nbr' in df.columns else df.iloc[:0]
        
        for feat in WASS_FEATURES:
            if feat in df.columns:
                client_vals = df.loc[df['patient_nbr'].isin(pid_set), feat].values if 'patient_nbr' in df.columns else np.array([])
                if len(client_vals) > 10:
                    w = wasserstein_distance(global_distributions[feat], client_vals)
                    wass_records.append({
                        'alpha': alpha, 'client': f'C{client_id}',
                        'feature': feat, 'wasserstein': w
                    })

wass_df_unfiltered = pd.DataFrame(wass_records)
print(wass_df_unfiltered.groupby(['alpha', 'feature'])['wasserstein'].mean().unstack('feature').round(4))

In [ ]:
if not wass_df_unfiltered.empty:
    fig, axes = plt.subplots(1, len(ALPHA_SWEEP), figsize=(5 * len(ALPHA_SWEEP), 5), sharey=True)
    if len(ALPHA_SWEEP) == 1:
        axes = [axes]
    fig.suptitle('Wasserstein Distance: Client vs Global Feature Distribution',
                 fontweight='bold', fontsize=13)

    for ax, alpha in zip(axes, ALPHA_SWEEP):
        sub = wass_df_unfiltered[wass_df_unfiltered['alpha'] == alpha]
        if sub.empty:
            ax.set_visible(False); continue
        pivot = sub.pivot(index='feature', columns='client', values='wasserstein')
        pivot.plot(kind='barh', ax=ax, colormap='Set2', alpha=0.85, edgecolor='white')
        ax.set_title(f'α = {alpha}', fontsize=12, fontweight='bold')
        ax.set_xlabel('W₁ distance from global')
        ax.set_ylabel('Feature' if ax == axes[0] else '')
        ax.legend(title='Client', fontsize=8, loc='lower right')

    plt.tight_layout()
    plt.savefig('figures/07_wasserstein_distances.png', bbox_inches='tight')
    plt.show()

    # ── Mean W1 per alpha ──────────────────────────────────────────────────────
    # NOTE: W₁ is NOT monotone in α for this dataset.
    # α is a Dirichlet draw parameter — any single realization may not respect
    # the expected heterogeneity ordering. W₁ measures the actual realized
    # divergence and is the reliable metric for the Pareto analysis.
    mean_wass = wass_df_unfiltered.groupby('alpha')['wasserstein'].mean()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(mean_wass.index, mean_wass.values, 'o-', color='steelblue',
            linewidth=2, markersize=8)

    # Annotate each point with its realized W₁ value
    for alpha_val, w1_val in mean_wass.items():
        ax.annotate(f'W₁={w1_val:.3f}',
                    xy=(alpha_val, w1_val),
                    xytext=(6, 6), textcoords='offset points',
                    fontsize=8, color='steelblue')

    ax.set_xlabel('Dirichlet α  (generation parameter, not realized heterogeneity)')
    ax.set_ylabel('Mean W₁ (across clients and features)')
    ax.set_title('Realized W₁ heterogeneity per partition\n'
                 'α is not a monotone proxy for W₁ in this dataset',
                 fontweight='bold')
    ax.invert_xaxis()

    # Identify and annotate the actual most heterogeneous partition
    most_het_alpha = mean_wass.idxmax()
    ax.annotate(f'Most heterogeneous\nrealized partition',
                xy=(most_het_alpha, mean_wass[most_het_alpha]),
                xytext=(-60, -30), textcoords='offset points',
                fontsize=8, color='#E24A33',
                arrowprops=dict(arrowstyle='->', color='#E24A33', lw=1.2))

    plt.tight_layout()
    plt.savefig('figures/08_mean_wasserstein_vs_alpha.png', bbox_inches='tight')
    plt.show()

    print('Realized heterogeneity ordering by W₁:')
    print(mean_wass.sort_values(ascending=False).to_string())
    print(f'\nNote: α ordering ({sorted(mean_wass.index, reverse=True)}) does not match '
          f'W₁ ordering ({list(mean_wass.sort_values(ascending=False).index)})')
    print('Use W₁ values to anchor the Pareto analysis x-axis, not α alone.')

## Label Shift vs Covariate Shift

## Filtered

In [ ]:
from scipy.spatial.distance import jensenshannon

# Global label distribution
global_label_dist = np.array([1 - y.mean(), y.mean()])  # [P(y=0), P(y=1)]

js_records = []

for alpha, partition in filtered_partitions.items():
    for client_id, patient_ids in partition.items():
        pid_set = set(patient_ids)
        
        # Client label distribution
        client_labels = [patient_labels_map.get(p, 0) for p in pid_set]
        n_pos = sum(client_labels)
        n_total = len(client_labels)
        client_label_dist = np.array([(n_total - n_pos) / n_total, n_pos / n_total])
        
        # Jensen-Shannon divergence (base-2 → range [0, 1])
        js = jensenshannon(global_label_dist, client_label_dist, base=2)
        
        # Mean W1 for this client (covariate shift)
        client_w1 = wass_df_filtered[
            (wass_df_filtered['alpha'] == alpha) & (wass_df_filtered['client'] == f'C{client_id}')
        ]['wasserstein'].mean()
        
        js_records.append({
            'alpha': alpha,
            'client': f'C{client_id}',
            'js_divergence': js,
            'mean_w1': client_w1,
            'pos_rate': n_pos / n_total,
            'n_patients': n_total
        })

js_df = pd.DataFrame(js_records)

# ── Plot: JS (label shift) vs W1 (covariate shift) side by side ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Label Shift vs Covariate Shift per Client', fontweight='bold', fontsize=13)

for ax, (metric, label, color) in zip(axes, [
    ('js_divergence', 'JS Divergence (label shift)',   'darkorange'),
    ('mean_w1',       'Mean W₁ (covariate shift)',     'steelblue'),
]):
    for alpha in ALPHA_SWEEP:
        sub = js_df[js_df['alpha'] == alpha]
        if sub.empty:
            continue
        ax.scatter(
            [str(alpha)] * len(sub), sub[metric],
            s=sub['n_patients'] / 500,   # bubble size = client size
            alpha=0.7, color=color, edgecolors='white', linewidth=0.8
        )
    ax.set_xlabel('α')
    ax.set_ylabel(label)
    ax.set_title(label)

plt.tight_layout()
plt.savefig('figures/11_label_vs_covariate_shift.png', bbox_inches='tight')
plt.show()

# ── Scatter: does label shift correlate with covariate shift? ─────────────────
fig, ax = plt.subplots(figsize=(7, 5))
for alpha in ALPHA_SWEEP:
    sub = js_df[js_df['alpha'] == alpha]
    ax.scatter(sub['js_divergence'], sub['mean_w1'],
               label=f'α={alpha}', s=60, alpha=0.8, edgecolors='white')

ax.set_xlabel('JS Divergence (label shift)')
ax.set_ylabel('Mean W₁ (covariate shift)')
ax.set_title('Are label shift and covariate shift correlated?\n'
             '(each point = one client)', fontweight='bold')
ax.legend(title='α', fontsize=9)
plt.tight_layout()
plt.savefig('figures/12_shift_correlation.png', bbox_inches='tight')
plt.show()

print(js_df.groupby('alpha')[['js_divergence', 'mean_w1']].mean().round(4))

## Unfiltered

In [ ]:
from scipy.spatial.distance import jensenshannon

# Global label distribution
global_label_dist = np.array([1 - y.mean(), y.mean()])  # [P(y=0), P(y=1)]

js_records = []

for alpha, partition in unfiltered_partitions.items():
    for client_id, patient_ids in partition.items():
        pid_set = set(patient_ids)
        
        # Client label distribution
        client_labels = [patient_labels_map.get(p, 0) for p in pid_set]
        n_pos = sum(client_labels)
        n_total = len(client_labels)
        client_label_dist = np.array([(n_total - n_pos) / n_total, n_pos / n_total])
        
        # Jensen-Shannon divergence (base-2 → range [0, 1])
        js = jensenshannon(global_label_dist, client_label_dist, base=2)
        
        # Mean W1 for this client (covariate shift)
        client_w1 = wass_df_unfiltered[
            (wass_df_unfiltered['alpha'] == alpha) & (wass_df_unfiltered['client'] == f'C{client_id}')
        ]['wasserstein'].mean()
        
        js_records.append({
            'alpha': alpha,
            'client': f'C{client_id}',
            'js_divergence': js,
            'mean_w1': client_w1,
            'pos_rate': n_pos / n_total,
            'n_patients': n_total
        })

js_df = pd.DataFrame(js_records)

# ── Plot: JS (label shift) vs W1 (covariate shift) side by side ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Label Shift vs Covariate Shift per Client', fontweight='bold', fontsize=13)

for ax, (metric, label, color) in zip(axes, [
    ('js_divergence', 'JS Divergence (label shift)',   'darkorange'),
    ('mean_w1',       'Mean W₁ (covariate shift)',     'steelblue'),
]):
    for alpha in ALPHA_SWEEP:
        sub = js_df[js_df['alpha'] == alpha]
        if sub.empty:
            continue
        ax.scatter(
            [str(alpha)] * len(sub), sub[metric],
            s=sub['n_patients'] / 500,   # bubble size = client size
            alpha=0.7, color=color, edgecolors='white', linewidth=0.8
        )
    ax.set_xlabel('α')
    ax.set_ylabel(label)
    ax.set_title(label)

plt.tight_layout()
plt.savefig('figures/11_label_vs_covariate_shift.png', bbox_inches='tight')
plt.show()

# ── Scatter: does label shift correlate with covariate shift? ─────────────────
fig, ax = plt.subplots(figsize=(7, 5))
for alpha in ALPHA_SWEEP:
    sub = js_df[js_df['alpha'] == alpha]
    ax.scatter(sub['js_divergence'], sub['mean_w1'],
               label=f'α={alpha}', s=60, alpha=0.8, edgecolors='white')

ax.set_xlabel('JS Divergence (label shift)')
ax.set_ylabel('Mean W₁ (covariate shift)')
ax.set_title('Are label shift and covariate shift correlated?\n'
             '(each point = one client)', fontweight='bold')
ax.legend(title='α', fontsize=9)
plt.tight_layout()
plt.savefig('figures/12_shift_correlation.png', bbox_inches='tight')
plt.show()

print(js_df.groupby('alpha')[['js_divergence', 'mean_w1']].mean().round(4))

## 10. Distribution of W1 Across Clients

## Filtered

In [ ]:
eff_records = []

for alpha, partition in filtered_partitions.items():
    for client_id, patient_ids in partition.items():
        pid_set = set(patient_ids)
        client_labels = [patient_labels_map.get(p, 0) for p in pid_set]
        n_total   = len(client_labels)
        n_pos     = sum(client_labels)
        n_neg     = n_total - n_pos
        pos_rate  = n_pos / n_total if n_total > 0 else 0

        eff_records.append({
            'alpha':    alpha,
            'client':   f'C{client_id}',
            'n_total':  n_total,
            'n_pos':    n_pos,           # effective minority-class size
            'n_neg':    n_neg,
            'pos_rate': pos_rate,
        })

eff_df = pd.DataFrame(eff_records)

# ── Plot n_pos per client across alpha values ──────────────────────────────────
fig, axes = plt.subplots(1, len(ALPHA_SWEEP), figsize=(5 * len(ALPHA_SWEEP), 5), sharey=False)
fig.suptitle('Effective Minority-Class Sample Size per Client\n'
             '(n_pos = patients ever readmitted within 30d)',
             fontweight='bold', fontsize=13)

# Reference: minimum viable sample size for learning
MIN_VIABLE = 200

for ax, alpha in zip(axes, ALPHA_SWEEP):
    sub = eff_df[eff_df['alpha'] == alpha].sort_values('client')
    colors = ['#E24A33' if n < MIN_VIABLE else '#4878CF' for n in sub['n_pos']]
    
    bars = ax.bar(sub['client'], sub['n_pos'], color=colors,
                  alpha=0.85, edgecolor='white')
    
    ax.axhline(MIN_VIABLE, color='black', linestyle='--', linewidth=1.2,
               label=f'Min viable ({MIN_VIABLE})')
    ax.set_title(f'α = {alpha}', fontweight='bold')
    ax.set_ylabel('# positive patients' if alpha == ALPHA_SWEEP[0] else '')
    ax.set_xlabel('Client')
    ax.legend(fontsize=8)
    
    for bar, n in zip(bars, sub['n_pos']):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + ax.get_ylim()[1] * 0.01,
                str(int(n)), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/14_effective_minority_size.png', bbox_inches='tight')
plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
print('Effective minority-class sample size per client:\n')
pivot = eff_df.pivot(index='client', columns='alpha', values='n_pos').astype(int)
print(pivot.to_string())
print(f'\nClients below {MIN_VIABLE} positive samples (unreliable local learning):')
print(eff_df[eff_df['n_pos'] < MIN_VIABLE][['alpha','client','n_pos','pos_rate']].to_string(index=False))

In [ ]:
for alpha in ALPHA_SWEEP: 

    print(f'\n{"─" * 65}')
    print(f'  α={alpha} client data summary:')
    print(f'  {"Client":<10} {"Train":>8} {"Val":>8} {"Test":>8} {"Features":>10} {"Pos. rate":>10}')
    print(f'  {"─"*10} {"─"*8} {"─"*8} {"─"*8} {"─"*10} {"─"*10}')
    client_info = {}

    for i in range(N_CLIENTS):
        client_dir = os.path.join(FILTERED_DIR, f'alpha_{alpha}', f'client_{i}')
        with open(os.path.join(client_dir, 'client_info.json'), 'r') as f:
            client_info[f'client_{i}'] = json.load(f)
            

    for k, v in client_info.items():
        print(f'  {k:<10} {v["n_train"]:>8,} {v["n_val"]:>8,} '
            f'{v["n_test"]:>8,} {v["n_features"]:>10,} {v["positive_rate"]:>10.3f}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Client Data Distribution', fontweight='bold')

    clients_keys     = list(client_info.keys())
    train_sizes = [client_info[c]['n_train'] for c in clients_keys]
    pos_rates   = [client_info[c]['positive_rate'] for c in clients_keys]

    axes[0].bar(clients_keys, train_sizes, color='steelblue')
    axes[0].set_title('Training set size per client')
    axes[0].set_ylabel('Samples')
    axes[0].tick_params(axis='x', rotation=15)

    axes[1].bar(clients_keys, pos_rates, color='darkorange')
    axes[1].axhline(sum(pos_rates) / len(pos_rates), color='gray',
                    linestyle='--', label='Mean')
    axes[1].set_title('Positive rate (30-day readmission) per client')
    axes[1].set_ylabel('Positive rate')
    axes[1].set_ylim(0, 0.25)
    axes[1].tick_params(axis='x', rotation=15)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(FILTERED_DIR, 'client_distribution.png'), dpi=150, bbox_inches='tight')
    plt.show()

## Unfiltered

In [ ]:
eff_records = []

for alpha, partition in unfiltered_partitions.items():
    for client_id, patient_ids in partition.items():
        pid_set = set(patient_ids)
        client_labels = [patient_labels_map.get(p, 0) for p in pid_set]
        n_total   = len(client_labels)
        n_pos     = sum(client_labels)
        n_neg     = n_total - n_pos
        pos_rate  = n_pos / n_total if n_total > 0 else 0

        eff_records.append({
            'alpha':    alpha,
            'client':   f'C{client_id}',
            'n_total':  n_total,
            'n_pos':    n_pos,           # effective minority-class size
            'n_neg':    n_neg,
            'pos_rate': pos_rate,
        })

eff_df = pd.DataFrame(eff_records)

# ── Plot n_pos per client across alpha values ──────────────────────────────────
fig, axes = plt.subplots(1, len(ALPHA_SWEEP), figsize=(5 * len(ALPHA_SWEEP), 5), sharey=False)
fig.suptitle('Effective Minority-Class Sample Size per Client\n'
             '(n_pos = patients ever readmitted within 30d)',
             fontweight='bold', fontsize=13)

# Reference: minimum viable sample size for learning
MIN_VIABLE = 200

for ax, alpha in zip(axes, ALPHA_SWEEP):
    sub = eff_df[eff_df['alpha'] == alpha].sort_values('client')
    colors = ['#E24A33' if n < MIN_VIABLE else '#4878CF' for n in sub['n_pos']]
    
    bars = ax.bar(sub['client'], sub['n_pos'], color=colors,
                  alpha=0.85, edgecolor='white')
    
    ax.axhline(MIN_VIABLE, color='black', linestyle='--', linewidth=1.2,
               label=f'Min viable ({MIN_VIABLE})')
    ax.set_title(f'α = {alpha}', fontweight='bold')
    ax.set_ylabel('# positive patients' if alpha == ALPHA_SWEEP[0] else '')
    ax.set_xlabel('Client')
    ax.legend(fontsize=8)
    
    for bar, n in zip(bars, sub['n_pos']):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + ax.get_ylim()[1] * 0.01,
                str(int(n)), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/14_effective_minority_size.png', bbox_inches='tight')
plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
print('Effective minority-class sample size per client:\n')
pivot = eff_df.pivot(index='client', columns='alpha', values='n_pos').astype(int)
print(pivot.to_string())
print(f'\nClients below {MIN_VIABLE} positive samples (unreliable local learning):')
print(eff_df[eff_df['n_pos'] < MIN_VIABLE][['alpha','client','n_pos','pos_rate']].to_string(index=False))

In [ ]:
for alpha in ALPHA_SWEEP: 

    print(f'\n{"─" * 65}')
    print(f'  α={alpha} client data summary:')
    print(f'  {"Client":<10} {"Train":>8} {"Val":>8} {"Test":>8} {"Features":>10} {"Pos. rate":>10}')
    print(f'  {"─"*10} {"─"*8} {"─"*8} {"─"*8} {"─"*10} {"─"*10}')
    client_info = {}

    for i in range(N_CLIENTS):
        client_dir = os.path.join(UNFILTERED_DIR, f'alpha_{alpha}', f'client_{i}')
        with open(os.path.join(client_dir, 'client_info.json'), 'r') as f:
            client_info[f'client_{i}'] = json.load(f)
            

    for k, v in client_info.items():
        print(f'  {k:<10} {v["n_train"]:>8,} {v["n_val"]:>8,} '
            f'{v["n_test"]:>8,} {v["n_features"]:>10,} {v["positive_rate"]:>10.3f}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Client Data Distribution', fontweight='bold')

    clients_keys     = list(client_info.keys())
    train_sizes = [client_info[c]['n_train'] for c in clients_keys]
    pos_rates   = [client_info[c]['positive_rate'] for c in clients_keys]

    axes[0].bar(clients_keys, train_sizes, color='steelblue')
    axes[0].set_title('Training set size per client')
    axes[0].set_ylabel('Samples')
    axes[0].tick_params(axis='x', rotation=15)

    axes[1].bar(clients_keys, pos_rates, color='darkorange')
    axes[1].axhline(sum(pos_rates) / len(pos_rates), color='gray',
                    linestyle='--', label='Mean')
    axes[1].set_title('Positive rate (30-day readmission) per client')
    axes[1].set_ylabel('Positive rate')
    axes[1].set_ylim(0, 0.25)
    axes[1].tick_params(axis='x', rotation=15)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(UNFILTERED_DIR, 'client_distribution.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 11. Per-Client Label Prior Distortion (Impact on Generator's p^(y))

In [ ]:
# Use only the alphas present in eff_df — avoids shape mismatch when
# some alphas from ALPHA_SWEEP were not found on disk (e.g. α=0.1).
available_alphas = sorted(eff_df['alpha'].unique())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("FedGen Label Prior Estimation: How Heterogeneity Distorts $\\hat{p}(y)$",
             fontweight='bold', fontsize=13)
true_pos_rate = y.mean()
prior_df = eff_df.groupby('alpha')['pos_rate'].mean().reset_index().rename(columns={'pos_rate': 'estimated_phat'})
prior_df['true_rate'] = true_pos_rate
prior_df['distortion'] = prior_df['estimated_phat'] - prior_df['true_rate']

# Left: estimated vs true prior per alpha
ax = axes[0]
plot_prior = prior_df[prior_df['alpha'].isin(available_alphas)].reset_index(drop=True)
ax.bar(plot_prior['alpha'].astype(str), plot_prior['estimated_phat'],
       color='steelblue', alpha=0.8, label='Estimated $\\hat{p}(y=1)$', width=0.4)
ax.axhline(true_pos_rate, color='#E24A33', linestyle='--', linewidth=1.5,
           label=f'True global rate ({true_pos_rate:.3f})')
ax.set_xlabel('α')
ax.set_ylabel('P(y=1)')
ax.set_title('Estimated label prior $\\hat{p}(y=1)$\nvs true global rate')
ax.legend(fontsize=9)
for i, row in plot_prior.iterrows():
    ax.text(i, row['estimated_phat'] + 0.002,
            f"{row['estimated_phat']:.3f}", ha='center', fontsize=9)

# Right: per-client positive rate vs global
ax = axes[1]
x     = np.arange(N_CLIENTS)
n_a   = len(available_alphas)
width = 0.8 / n_a   # divide bar group width evenly across available alphas

for i, alpha in enumerate(available_alphas):
    sub = eff_df[eff_df['alpha'] == alpha].sort_values('client').reset_index(drop=True)
    ax.bar(x + i * width, sub['pos_rate'], width=width,
           alpha=0.8, label=f'α={alpha}', edgecolor='white')

ax.axhline(true_pos_rate, color='black', linestyle='--',
           linewidth=1.2, label=f'Global ({true_pos_rate:.3f})')
ax.set_xticks(x + width * (n_a - 1) / 2)
ax.set_xticklabels([f'C{i}' for i in range(N_CLIENTS)])
ax.set_ylabel('Positive rate')
ax.set_title('Per-client positive rate by α\n(drives $\\hat{p}(y)$ estimation in FedGen)')
ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('figures/15_label_prior_distortion.png', bbox_inches='tight')
plt.show()

print('\nLabel prior distortion summary:')
print(plot_prior[['alpha', 'estimated_phat', 'true_rate', 'distortion']].to_string(index=False))

In [ ]:
# Use only the alphas present in eff_df — avoids shape mismatch when
# some alphas from ALPHA_SWEEP were not found on disk (e.g. α=0.1).
available_alphas = sorted(eff_df['alpha'].unique())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("FedGen Label Prior Estimation: How Heterogeneity Distorts $\\hat{p}(y)$",
             fontweight='bold', fontsize=13)
true_pos_rate = y.mean()
prior_df = eff_df.groupby('alpha')['pos_rate'].mean().reset_index().rename(columns={'pos_rate': 'estimated_phat'})
prior_df['true_rate'] = true_pos_rate
prior_df['distortion'] = prior_df['estimated_phat'] - prior_df['true_rate']

# Left: estimated vs true prior per alpha
ax = axes[0]
plot_prior = prior_df[prior_df['alpha'].isin(available_alphas)].reset_index(drop=True)
ax.bar(plot_prior['alpha'].astype(str), plot_prior['estimated_phat'],
       color='steelblue', alpha=0.8, label='Estimated $\\hat{p}(y=1)$', width=0.4)
ax.axhline(true_pos_rate, color='#E24A33', linestyle='--', linewidth=1.5,
           label=f'True global rate ({true_pos_rate:.3f})')
ax.set_xlabel('α')
ax.set_ylabel('P(y=1)')
ax.set_title('Estimated label prior $\\hat{p}(y=1)$\nvs true global rate')
ax.legend(fontsize=9)
for i, row in plot_prior.iterrows():
    ax.text(i, row['estimated_phat'] + 0.002,
            f"{row['estimated_phat']:.3f}", ha='center', fontsize=9)

# Right: per-client positive rate vs global
ax = axes[1]
x     = np.arange(N_CLIENTS)
n_a   = len(available_alphas)
width = 0.8 / n_a   # divide bar group width evenly across available alphas

for i, alpha in enumerate(available_alphas):
    sub = eff_df[eff_df['alpha'] == alpha].sort_values('client').reset_index(drop=True)
    ax.bar(x + i * width, sub['pos_rate'], width=width,
           alpha=0.8, label=f'α={alpha}', edgecolor='white')

ax.axhline(true_pos_rate, color='black', linestyle='--',
           linewidth=1.2, label=f'Global ({true_pos_rate:.3f})')
ax.set_xticks(x + width * (n_a - 1) / 2)
ax.set_xticklabels([f'C{i}' for i in range(N_CLIENTS)])
ax.set_ylabel('Positive rate')
ax.set_title('Per-client positive rate by α\n(drives $\\hat{p}(y)$ estimation in FedGen)')
ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('figures/15_label_prior_distortion.png', bbox_inches='tight')
plt.show()

print('\nLabel prior distortion summary:')
print(plot_prior[['alpha', 'estimated_phat', 'true_rate', 'distortion']].to_string(index=False))

# Gini analysis

In [ ]:
from scipy.stats import entropy as scipy_entropy
from scipy.spatial.distance import jensenshannon

def gini(values):
    """
    Gini coefficient of an array of non-negative values.
    0 = perfect equality, 1 = maximum inequality.
    Classic measure of distributional imbalance.
    """
    v = np.sort(np.array(values, dtype=float))
    n = len(v)
    if v.sum() == 0:
        return 0.0
    return (2 * np.sum(np.arange(1, n + 1) * v) / (n * v.sum())) - (n + 1) / n


def total_variation(p, q):
    """
    Total variation distance between two probability distributions.
    TV = 0.5 * sum |p_i - q_i|. Range [0, 1].
    More interpretable than KL: TV=0.2 means 20% of probability mass differs.
    """
    p, q = np.array(p), np.array(q)
    return 0.5 * np.sum(np.abs(p - q))


def effective_clients(sizes):
    """
    Effective number of clients = exp(entropy of size distribution).
    If all clients are equal size → returns N_CLIENTS.
    If one client dominates → approaches 1.
    Captures data quantity imbalance independently of label shift.
    """
    s = np.array(sizes, dtype=float)
    p = s / s.sum()
    return np.exp(scipy_entropy(p))


In [ ]:


global_label_dist = np.array([1 - y.mean(), y.mean()])

het_records = []

for alpha, partition in filtered_partitions.items():
    sizes     = [len(v) for v in partition.values()]
    pos_rates = [
        sum(patient_labels_map.get(p, 0) for p in v) / len(v)
        for v in partition.values()
    ]

    # ── 1. Gini on client sizes (quantity heterogeneity) ─────────────────────
    gini_size = gini(sizes)

    # ── 2. Gini on positive rates (label heterogeneity) ──────────────────────
    gini_label = gini(pos_rates)

    # ── 3. Coefficient of variation of positive rates ─────────────────────────
    # CV = std/mean. Scale-free measure of label rate dispersion.
    cv_label = np.std(pos_rates) / np.mean(pos_rates)

    # ── 4. Mean JS divergence of per-client label dist from global ────────────
    js_vals = []
    for pos_rate in pos_rates:
        client_dist = np.array([1 - pos_rate, pos_rate])
        js_vals.append(jensenshannon(global_label_dist, client_dist, base=2))
    mean_js = np.mean(js_vals)
    max_js  = np.max(js_vals)

    # ── 5. Mean total variation distance from global label dist ───────────────
    tv_vals = [
        total_variation(global_label_dist, [1 - r, r])
        for r in pos_rates
    ]
    mean_tv = np.mean(tv_vals)

    # ── 6. Mean W₁ on features (already computed in wass_df) ─────────────────
    mean_w1 = wass_df_filtered[wass_df_filtered['alpha'] == alpha]['wasserstein'].mean() \
              if not wass_df_filtered.empty else np.nan

    # ── 7. Effective number of clients ────────────────────────────────────────
    eff_n = effective_clients(sizes)

    het_records.append({
        'α':              alpha,
        'Gini (sizes)':   round(gini_size,  4),
        'Gini (labels)':  round(gini_label, 4),
        'CV (labels)':    round(cv_label,   4),
        'Mean JS':        round(mean_js,    4),
        'Max JS':         round(max_js,     4),
        'Mean TV':        round(mean_tv,    4),
        'Mean W₁':        round(mean_w1,    4),
        'Eff. clients':   round(eff_n,      2),
    })

het_df_filtered = pd.DataFrame(het_records).set_index('α')

print('Heterogeneity measures per partition')
print('='*75)
print(het_df_filtered.to_string())
print()
print('Rank order per metric (most heterogeneous = rank 1):')
print('-'*75)
rank_df = het_df_filtered.copy()
# For Eff. clients: lower = more heterogeneous, so invert rank
for col in rank_df.columns:
    ascending = True if col == 'Eff. clients' else False
    rank_df[col] = rank_df[col].rank(ascending=ascending).astype(int)
print(rank_df.to_string())
print()
# Composite rank: mean rank across all metrics
het_df_filtered['Composite rank'] = rank_df.mean(axis=1).round(2)
print('Composite heterogeneity score (lower = more heterogeneous):')
print(het_df_filtered['Composite rank'].sort_values().to_string())

In [ ]:


global_label_dist = np.array([1 - y.mean(), y.mean()])

het_records = []

for alpha, partition in unfiltered_partitions.items():
    sizes     = [len(v) for v in partition.values()]
    pos_rates = [
        sum(patient_labels_map.get(p, 0) for p in v) / len(v)
        for v in partition.values()
    ]

    # ── 1. Gini on client sizes (quantity heterogeneity) ─────────────────────
    gini_size = gini(sizes)

    # ── 2. Gini on positive rates (label heterogeneity) ──────────────────────
    gini_label = gini(pos_rates)

    # ── 3. Coefficient of variation of positive rates ─────────────────────────
    # CV = std/mean. Scale-free measure of label rate dispersion.
    cv_label = np.std(pos_rates) / np.mean(pos_rates)

    # ── 4. Mean JS divergence of per-client label dist from global ────────────
    js_vals = []
    for pos_rate in pos_rates:
        client_dist = np.array([1 - pos_rate, pos_rate])
        js_vals.append(jensenshannon(global_label_dist, client_dist, base=2))
    mean_js = np.mean(js_vals)
    max_js  = np.max(js_vals)

    # ── 5. Mean total variation distance from global label dist ───────────────
    tv_vals = [
        total_variation(global_label_dist, [1 - r, r])
        for r in pos_rates
    ]
    mean_tv = np.mean(tv_vals)

    # ── 6. Mean W₁ on features (already computed in wass_df) ─────────────────
    mean_w1 = wass_df_unfiltered[wass_df_unfiltered['alpha'] == alpha]['wasserstein'].mean() \
              if not wass_df_unfiltered.empty else np.nan

    # ── 7. Effective number of clients ────────────────────────────────────────
    eff_n = effective_clients(sizes)

    het_records.append({
        'α':              alpha,
        'Gini (sizes)':   round(gini_size,  4),
        'Gini (labels)':  round(gini_label, 4),
        'CV (labels)':    round(cv_label,   4),
        'Mean JS':        round(mean_js,    4),
        'Max JS':         round(max_js,     4),
        'Mean TV':        round(mean_tv,    4),
        'Mean W₁':        round(mean_w1,    4),
        'Eff. clients':   round(eff_n,      2),
    })

het_df_unfiltered = pd.DataFrame(het_records).set_index('α')

print('Heterogeneity measures per partition')
print('='*75)
print(het_df_unfiltered.to_string())
print()
print('Rank order per metric (most heterogeneous = rank 1):')
print('-'*75)
rank_df = het_df_unfiltered.copy()
# For Eff. clients: lower = more heterogeneous, so invert rank
for col in rank_df.columns:
    ascending = True if col == 'Eff. clients' else False
    rank_df[col] = rank_df[col].rank(ascending=ascending).astype(int)
print(rank_df.to_string())
print()
# Composite rank: mean rank across all metrics
het_df_unfiltered['Composite rank'] = rank_df.mean(axis=1).round(2)
print('Composite heterogeneity score (lower = more heterogeneous):')
print(het_df_unfiltered['Composite rank'].sort_values().to_string())